In [ ]:
import os
import sys
import json
import time

In [ ]:
# Once the raw data is placed as described above, set the path to the TabFormer directory

# Change this path to point to TabFormer data
data_root_dir = os.path.abspath('../data/TabFormer/') 
# Change this path to the directory where you want to save your model
model_output_dir = os.path.join(data_root_dir, 'trained_models_np')

# Path to save the trained model
os.makedirs(model_output_dir, exist_ok=True)

In [ ]:
def print_tree(directory, prefix=""):
    """Recursively prints the directory tree starting at 'directory'."""
    # Retrieve a sorted list of entries in the directory
    entries = sorted(os.listdir(directory))
    entries_count = len(entries)
    
    for index, entry in enumerate(entries):
        path = os.path.join(directory, entry)
        # Determine the branch connector
        if index == entries_count - 1:
            connector = "└── "
            extension = "    "
        else:
            connector = "├── "
            extension = "│   "
        
        print(prefix + connector + entry)
        
        # If the entry is a directory, recursively print its contents
        if os.path.isdir(path):
            print_tree(path, prefix + extension)

In [ ]:
# Check if the raw data has been placed properly
print_tree(data_root_dir)

In [ ]:
# Add the "src" directory to the search path
src_dir = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), 'src'))
sys.path.insert(0, src_dir)

# Import node prediction preprocessing functions
from preprocess_TabFormer_np import preprocess_data, load_hetero_graph

In [ ]:
# Preprocess the data for node prediction
print("Preprocessing for Node Prediction...")
user_mask_map, mx_mask_map, tx_mask_map = preprocess_data(data_root_dir)

# This will output correlation statistics

In [ ]:
# View the created directory structure
print("Node Prediction data structure:")
print_tree(os.path.join(data_root_dir, 'gnn_np'))

In [ ]:
from preprocess_TabFormer_np import plot_graph_structure

test_data = load_hetero_graph(os.path.join(data_root_dir, "gnn_np", "test_gnn"))
plot_graph_structure(os.path.join(data_root_dir, "gnn_np", "test_gnn"))

In [ ]:
training_config = {
    "paths": {
        "data_dir": "/data",  # Mount gnn_np dataset directory under /data
        "output_dir": "/trained_models"  # Path to save the trained models
    },
    "models": [
        {
            "kind": "GNN_XGBoost_NP",  # ⭐ Node Prediction model type
            "gpu": "single",
            "hyperparameters": {
                "gnn": {
                    "hidden_channels": 32,
                    "n_hops": 2,
                    "layer": "TransformerConv",
                    "dropout_prob": 0.2,
                    "batch_size": 4096,
                    "fan_out": 8,
                    "metric": "f1",
                    "num_epochs": 10,
                    "weight_decay": 0.00001
                },
                "xgb": {
                    "max_depth": 3,
                    "learning_rate": 0.1,
                    "num_parallel_tree": 50,
                    "num_boost_round": 100,
                    "gamma": 0.0
                }
            }
        }
    ]
}


In [ ]:
training_config = {
    "paths": {
        "data_dir": "/data",
        "output_dir": "/trained_models"
    },
    "models": [
        {
            "kind": "GNN_XGBoost_NP",
            "gpu": "single",
            "hyperparameters": {
                "gnn": {
                    "hidden_channels": 16,
                    "n_hops": 2,
                    "dropout_prob": 0.2,
                    "batch_size": 4096,
                    "fan_out": 8,
                    "layer": "GATConv",
                    "metric": "f1",
                    "num_epochs": 10,
                    "weight_decay": 0.00001
                },
                "xgb": {
                    "max_depth": 5,
                    "num_parallel_tree": 50,
                    "num_boost_round": 100,
                    "learning_rate": 0.1,
                    "gamma": 0.0
                }
            }
        }
    ]
}



In [ ]:
training_config_file_name = 'training_config_np.json'

with open(os.path.join(training_config_file_name), 'w') as json_file:
    json.dump(training_config, json_file, indent=4)
    
print(f"✅ Training configuration saved to: {training_config_file_name}")

In [ ]:
CONTAINER_NAME = "financial-fraud-training-np"
gnn_data_dir = os.path.join(data_root_dir, "gnn_np")  # ⭐ Use gnn_np for node prediction

In [ ]:
# Stop any running container with the same name
container_ids = !docker ps --filter "name={CONTAINER_NAME}" -q
if len(container_ids) > 0:
    !docker stop {CONTAINER_NAME}

In [ ]:
if BREV:
    host_path_gnn_data = gnn_data_dir.replace('/root/verb-workspace', '/home/ubuntu/workspace')
    host_path_trained_models = model_output_dir.replace('/root/verb-workspace', '/home/ubuntu/workspace')
else:
    host_path_gnn_data = gnn_data_dir
    host_path_trained_models = model_output_dir

In [ ]:

# API_KEY="NGC API KEY"
# !echo "$API_KEY" | docker login nvcr.io --username '$oauthtoken' --password-stdin
# !docker pull nvcr.io/nvidia/cugraph/financial-fraud-training:2.0.0-rc5-amd64

In [ ]:
config_path = !realpath training_config_np.json
if BREV:
    host_config_path = config_path[0].replace('/root/verb-workspace', '/home/ubuntu/workspace')
else:
    host_config_path = config_path[0]
host_config_path

In [ ]:
# Check if the config file is properly mounted
!docker run  --entrypoint /bin/bash  -it --rm --name={CONTAINER_NAME} --gpus "device=0" -v {host_path_gnn_data}:/data \
    -v {host_path_trained_models}:/trained_models -v {host_config_path}:/app/config.json \
        nvcr.io/nvidia/cugraph/financial-fraud-training:2.0.0 -c "cat /app/config.json"

In [ ]:
!docker run -it --rm --name={CONTAINER_NAME} --gpus "device=0" -v {host_path_gnn_data}:/data \
    -v {host_path_trained_models}:/trained_models -v {host_config_path}:/app/config.json \
    nvcr.io/nvidia/cugraph/financial-fraud-training:2.0.0

In [ ]:
model_output_dir

In [ ]:
# Verify model repository structure
# model_repo_path = os.path.join(model_output_dir, 'model_repository')
model_repo_path = os.path.join(model_output_dir, 'python_backend_model_repository_np')
if os.path.exists(model_repo_path):
    print("✅ Training completed successfully!")
    print("\nModel repository structure:")
    print_tree(model_repo_path)
else:
    print("❌ Model repository not found. Training may have failed.")

In [ ]:
!pip install 'tritonclient[all]'

In [ ]:
import tritonclient.grpc as triton_grpc
import tritonclient.http as httpclient
from tritonclient import utils as triton_utils


In [ ]:
HTTP_PORT = 8000
GRPC_PORT = 8001
METRICS_PORT = 8002

In [ ]:
# Stop and remove any existing Triton container
container_ids = !docker ps -a --filter "name=tritonserver" -q
if len(container_ids) > 0:
    !docker stop tritonserver
    !docker rm tritonserver

In [ ]:
MODEL_REPO_PATH = os.path.join(model_output_dir, 'python_backend_model_repository_np')
MODEL_REPO_PATH


In [ ]:
#### Start Triton Server

In [ ]:
# Build a new image based on NVIDIA Triton, with gpu enabled XGBoost and Captum for computing Shapley values
TRITON_IMAGE = 'triton-with-gpu-xgboost'
!docker build -q -t {TRITON_IMAGE} ../triton


In [ ]:
# Serve trained model with NVIDIA Triton
if BREV:
    HOST_MODEL_REPO_PATH = MODEL_REPO_PATH.replace('/root/verb-workspace', '/home/ubuntu/workspace')
else:
    HOST_MODEL_REPO_PATH = MODEL_REPO_PATH

!docker run --gpus "device=0" -d -p {HTTP_PORT}:{HTTP_PORT} -p {GRPC_PORT}:{GRPC_PORT} \
    -v {HOST_MODEL_REPO_PATH}:/models --name tritonserver {TRITON_IMAGE} tritonserver \
    --model-repository=/models --exit-timeout-secs=6000 --http-port={HTTP_PORT} --grpc-port={GRPC_PORT} \
    --metrics-port={METRICS_PORT}

In [ ]:
client_grpc = triton_grpc.InferenceServerClient(url=f'{HOST}:{GRPC_PORT}')
client_http = httpclient.InferenceServerClient(url=f'{HOST}:{HTTP_PORT}')

In [ ]:
import subprocess
container_name = "tritonserver"

print("Waiting for Triton server to be ready...\\n")

while True:
    client_grpc = triton_grpc.InferenceServerClient(url=f'{HOST}:{GRPC_PORT}')
    try:
        if client_grpc.is_server_ready():
            print("\\n✅ Triton server is ready!")
            break
    except triton_utils.InferenceServerException as e:
        pass
    try:
        output = subprocess.check_output(["docker", "logs", "--tail", "10", container_name])
        print(output.decode("utf-8"))
    except subprocess.CalledProcessError as e:
        print("Error retrieving logs:", e)
    time.sleep(10)

In [ ]:
!docker logs tritonserver

In [ ]:


import numpy as np

from tritonclient.http import InferenceServerClient, InferInput, InferRequestedOutput


In [ ]:


def make_example_request():
    # -- example sizes --
    num_merchants = 5
    num_users   = 7
    num_edges   = 3
    merchant_feature_dim = 24
    user_feature_dim = 13
    user_to_merchant_feature_dim = 38

    # -- 1) features --
    x_merchant = np.random.randn(num_merchants, merchant_feature_dim).astype(np.float32)
    x_user   = np.random.randn(num_users, user_feature_dim).astype(np.float32)

    # -- 2) shap flag and masks --
    compute_shap          = np.array([True], dtype=np.bool_)
    feature_mask_merchant   = np.random.randint(0,2, size=(merchant_feature_dim,), dtype=np.int32)
    feature_mask_user     = np.random.randint(0,2, size=(user_feature_dim,), dtype=np.int32)

    # -- 3) edges: index [2, num_edges] and attributes [num_edges,user_to_merchant_feature_dim] --
    edge_index_user_to_merchant = np.vstack([
        np.random.randint(0, num_users,   size=(num_edges,)),
        np.random.randint(0, num_merchants, size=(num_edges,))
    ]).astype(np.int64)
    
    edge_attr_user_to_merchant = np.random.randn(num_edges, user_to_merchant_feature_dim).astype(np.float32)

    feature_mask_user_to_merchant =  np.random.randint(0,2, size=(user_to_merchant_feature_dim,), dtype=np.int32)

    return {
        "x_merchant": x_merchant,
        "x_user": x_user,
        "COMPUTE_SHAP": compute_shap,
        "feature_mask_merchant": feature_mask_merchant,
        "feature_mask_user": feature_mask_user,
        "edge_index_user_to_merchant": edge_index_user_to_merchant,
        "edge_attr_user_to_merchant": edge_attr_user_to_merchant,
        "edge_feature_mask_user_to_merchant": feature_mask_user_to_merchant
    }



In [ ]:


def prepare_and_send_inference_request(data):

    # Connect to Triton
    client = httpclient.InferenceServerClient(url=f'{HOST}:{HTTP_PORT}')

    # Prepare Inputs

    inputs = []
    def _add_input(name, arr, dtype):
        inp = InferInput(name, arr.shape, datatype=dtype)
        inp.set_data_from_numpy(arr)
        inputs.append(inp)

    for key, value in data.items():
        if key.startswith("x_"):
            dtype = "FP32"
        elif key.startswith("feature_mask_"):
            dtype = "INT32"
        elif key.startswith("edge_feature_mask_"):
            dtype = "INT32"            
        elif key.startswith("edge_index_"):
            dtype = "INT64"
        elif key.startswith("edge_attr_"):
            dtype = "FP32"
        elif key == "COMPUTE_SHAP":
            dtype = "BOOL"
        else:
            continue  # skip things we don't care about

        _add_input(key, value, dtype)


    # Outputs

    outputs = [InferRequestedOutput("PREDICTION")]

    for key in data:
        if key.startswith("x_"):
            node = key[len("x_"):]  # extract node name
            outputs.append(InferRequestedOutput(f"shap_values_{node}"))
        elif key.startswith("edge_attr_"):
            edge_name = key[len("edge_attr_"):]  # extract edge name
            outputs.append(InferRequestedOutput(f"shap_values_{edge_name}"))
    
    # Send request

    model_name="prediction_and_shapley_np"
    response = client.infer(
        model_name,
        inputs=inputs,
        request_id=str(1),
        outputs=outputs,
        timeout= 3000
    )

    result = {}

    # always include prediction
    result["PREDICTION"] = response.as_numpy("PREDICTION")

    # add shap values
    for key in data:
        if key.startswith("x_"):
            node = key[len("x_"):]  # e.g. "merchant", "user"
            result[f"shap_values_{node}"] = response.as_numpy(f"shap_values_{node}")
        if key.startswith("edge_attr_"):
            edge_name = key[len("edge_attr_"):]  # e.g. ("user" "to"  "merchant")
            result[f"shap_values_{edge_name}"] = response.as_numpy(f"shap_values_{edge_name}")
    
    return result


In [ ]:
test_data =  load_hetero_graph(os.path.join(gnn_data_dir, "test_gnn"))
for k in test_data.keys():
    print(k)

In [ ]:

compute_shap = False
result =  prepare_and_send_inference_request(test_data | {"COMPUTE_SHAP": np.array([compute_shap], dtype=np.bool_)})

In [ ]:
result['PREDICTION']

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score)
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay


In [ ]:

import pandas as pd
def compute_score_for_batch(y, predictions, decision_threshold = 0.5):
    # Apply threshold
    y_pred = (predictions > decision_threshold).astype(int)

    # Compute evaluation metrics
    accuracy = accuracy_score(y, y_pred)
    precision = precision_score(y, y_pred, zero_division=0)
    recall = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)

    # Confusion matrix
    classes = ['Non-Fraud', 'Fraud']
    columns = pd.MultiIndex.from_product([["Predicted"], classes])
    index = pd.MultiIndex.from_product([["Actual"], classes])

    conf_mat = confusion_matrix(y, y_pred)
    cm_df = pd.DataFrame(conf_mat, index=index, columns=columns)
    print(cm_df)

    # Plot the confusion matrix directly
    disp = ConfusionMatrixDisplay.from_predictions(
        y, y_pred, display_labels=classes
    )
    disp.ax_.set_title('Confusion Matrix')
    plt.show()

    # Print summary
    print("----Summary---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")


In [ ]:
# Decision threshold to flag a transaction as fraud
#Change to trade-off precision and recall
decision_threshold = 0.5
y = test_data['node_label_transaction']
compute_score_for_batch(y, result['PREDICTION'], decision_threshold)

In [ ]:
compute_shap = True
result_with_shap = prepare_and_send_inference_request(test_data | {"COMPUTE_SHAP": np.array([compute_shap], dtype=np.bool_)})

In [ ]:
for key in result_with_shap:
    if key.startswith('shap_'):
        print(f'{key} : {result_with_shap[key]}')

In [ ]:
feature_masks = {
    'user': user_mask_map,
    'merchant': mx_mask_map,
    'transaction': tx_mask_map
}

In [ ]:
for key in feature_masks:
    shap_values = result_with_shap[f'shap_values_{key}']
    min_idx = min(feature_masks[key].values())

    attr_to_shap = {
        attr: float(shap_values[int(idx - min_idx)])
        for attr, idx in feature_masks[key].items()
    }
    print(attr_to_shap)